# Quick Start: Compressible Flow

This notebook demonstrates a minimal transient compressible flow setup: inviscid flow around
a cylinder at Mach 0.3 using HDG with implicit Euler time integration.

In [ ]:
import ngsolve as ngs
ngs.ngsglobals.msg_level = 0
from dream.mesh import get_cylinder_omesh
from dream.compressible_flow import CompressibleFlowSolver, FarField, InviscidWall, Initial

In [ ]:
# Structured O-mesh around cylinder: inner radius 0.5, outer radius 15
mesh = get_cylinder_omesh(ri=0.5, ro=15, n_polar=16, n_radial=8, geom=1.5)

In [ ]:
solver = CompressibleFlowSolver(mesh)

# Physical parameters
solver.mach_number = 0.3
solver.scaling     = 'aerodynamic'

# Discretisation
solver.fem            = 'conservative_hdg'
solver.fem.order      = 2
solver.riemann_solver = 'roe'

# Time integration
solver.time                = 'transient'
solver.fem.scheme          = 'implicit_euler'
solver.time.timer.interval = (0, 0.5)
solver.time.timer.step     = 5e-3

# Boundary and initial conditions
Uinf = solver.get_farfield_fields((1, 0))
solver.bcs['left|right'] = FarField(fields=Uinf)
solver.bcs['cylinder']   = InviscidWall()
solver.dcs['default']    = Initial(fields=Uinf)

# with ngs.TaskManager():
#     # Sets up finite element spaces, assembles bilinear forms and applies boundary conditions
#     solver.initialize()
#     # Runs the time-stepping loop defined by solver.time
#     solver.solve()